# Russian DPO with Soup on a Colab T4

Select **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
This notebook launches the repository's experiment runner in a fresh subprocess.
It captures setup commands, raw output, UTC timestamps, the environment, and the
experiment artifacts. A new attempt preserves earlier failures.

The 500 preference pairs are translated general Russian dialogue, not company
support tickets. The production verdict remains **DON'T SHIP** until reviewed
support-domain evidence and the independent verification gates pass. An
unexecuted notebook contains no GPU results.

In [ ]:
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import uuid

REPO_URL = "https://github.com/Huzyefahwasim/Soup.git"
REPO_REF = "main"  # Set an immutable commit SHA to replay a specific submission.
REPO = Path("/content/soup-assignment")
RUN_ID = datetime.now(timezone.utc).strftime("t4-%Y%m%dT%H%M%SZ-") + uuid.uuid4().hex[:8]
EVIDENCE_ROOT = Path("/content/soup-evidence") / RUN_ID
SETUP_DIR = EVIDENCE_ROOT / "setup"
SETUP_DIR.mkdir(parents=True, exist_ok=False)
RUN_DIR = REPO / "artifacts" / RUN_ID  # The runner must create this directory itself.
COMMANDS = []

os.environ.update(
    PYTHONUNBUFFERED="1",
    TOKENIZERS_PARALLELISM="false",
    HF_HUB_DISABLE_TELEMETRY="1",
    WANDB_DISABLED="true",
)
print("Fresh attempt:", RUN_ID)
print("Setup evidence:", SETUP_DIR)
print("Experiment artifacts:", RUN_DIR)

In [ ]:
def utc():
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

def logged(name, argv, *, cwd=None, require_success=True):
    label = f"{len(COMMANDS):02d}-{name}"
    record = {"name": name, "argv": [str(item) for item in argv],
              "cwd": str(cwd) if cwd is not None else None, "started_utc": utc()}
    record_path = SETUP_DIR / f"{label}.command.json"
    record_path.write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")
    with (SETUP_DIR / f"{label}.raw.log").open("xb") as raw, \
         (SETUP_DIR / f"{label}.timestamped.log").open("x", encoding="utf-8") as stamped:
        try:
            process = subprocess.Popen(
                record["argv"], cwd=cwd, stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT, env=dict(os.environ),
            )
        except OSError as error:
            record.update(returncode=127, launch_error=str(error))
            message = (str(error) + "\n").encode("utf-8")
            raw.write(message)
            stamped.write(f"[{utc()}] " + message.decode("utf-8"))
        else:
            for line in iter(process.stdout.readline, b""):
                raw.write(line)
                raw.flush()
                rendered = f"[{utc()}] " + line.decode("utf-8", errors="replace")
                stamped.write(rendered)
                stamped.flush()
                print(rendered, end="", flush=True)
            record["returncode"] = process.wait()
            process.stdout.close()
    record["finished_utc"] = utc()
    record_path.write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")
    COMMANDS.append(record)
    (SETUP_DIR / "commands.json").write_text(json.dumps(COMMANDS, indent=2) + "\n", encoding="utf-8")
    if require_success and record["returncode"] != 0:
        raise RuntimeError(f"{name} exited {record['returncode']}; inspect {record_path}")
    return record

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

## Check out the public repository

`REPO_REF="main"` resolves the latest public submission and records its commit.
For an exact replay, replace it with the commit recorded in a previous archive.
An existing checkout must be clean and have the same origin; this cell refuses
to reset local changes.

In [ ]:
if REPO.exists():
    if not (REPO / ".git").exists():
        raise RuntimeError(f"{REPO} exists but is not a Git checkout")
    origin = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO, text=True
    ).strip()
    if origin != REPO_URL:
        raise RuntimeError(f"Unexpected origin: {origin}")
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain", "--untracked-files=no"], cwd=REPO, text=True
    )
    if dirty:
        raise RuntimeError("The existing checkout has changes. Use a fresh runtime or preserve them first.")
else:
    logged("clone", ["git", "clone", "--no-checkout", "--depth", "1", REPO_URL, str(REPO)])

logged("fetch-ref", ["git", "fetch", "--depth", "1", "origin", REPO_REF], cwd=REPO)
logged("checkout-ref", ["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO)
logged("source-revision", ["git", "log", "-1", "--format=%H%n%cI%n%s"], cwd=REPO)
COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO, text=True
).strip()
identity = {
    "repo": REPO_URL, "requested_ref": REPO_REF, "resolved_commit": COMMIT,
    "run_id": RUN_ID, "started_utc": utc(), "python": sys.version,
    "requirements_sha256": sha256_file(REPO / "requirements-colab.txt"),
    "config_sha256": sha256_file(REPO / "configs" / "soup.yaml"),
    "data_manifest_sha256": sha256_file(REPO / "data" / "manifest.json"),
}
(SETUP_DIR / "source-identity.json").write_text(
    json.dumps(identity, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(identity, indent=2))

## Install the pinned training stack and record the environment

The requirements pin Soup 0.75.2, Transformers 5.16.1, TRL 0.29.0, and PEFT
0.20.0. Keep Colab's CUDA-enabled PyTorch; record its exact version and the full
resolved package list. Python must be 3.10 through 3.12.

Colab can include TorchAO 0.10.0. PEFT 0.20 rejects that optional package even
while wrapping ordinary unquantized LoRA layers. The cell below removes an
installed TorchAO older than 0.16.0 only after confirming this configuration
uses no quantization or QAT. The uninstall command and output remain in the
setup evidence. Fresh subprocesses avoid reusing an imported package.

In [ ]:
logged("nvidia-smi-before-setup", ["nvidia-smi"])
logged("python-before-setup", [sys.executable, "--version"])
logged("packages-before-setup", [sys.executable, "-m", "pip", "freeze"])
logged("install", [
    sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements-colab.txt")
])

import yaml
config = yaml.safe_load((REPO / "configs" / "soup.yaml").read_text(encoding="utf-8"))
training = config["training"]
assert training.get("quantization") == "none"
assert not training.get("quantization_aware", False)

try:
    torchao_version = importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    torchao_version = None
torchao_action = {"installed_version": torchao_version, "removed": False,
                  "reason": "Unquantized LoRA with PEFT 0.20; QAT disabled"}
if torchao_version is not None:
    match = re.match(r"^(\d+)\.(\d+)", torchao_version)
    if match is None:
        raise RuntimeError(f"Cannot compare TorchAO version: {torchao_version}")
    if tuple(map(int, match.groups())) < (0, 16):
        logged("remove-incompatible-unused-torchao", [
            sys.executable, "-m", "pip", "uninstall", "-y", "torchao"
        ])
        torchao_action["removed"] = True
(SETUP_DIR / "torchao-action.json").write_text(
    json.dumps(torchao_action, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(torchao_action, indent=2))
pip_check = logged("pip-check", [sys.executable, "-m", "pip", "check"], require_success=False)
logged("packages-after-setup", [sys.executable, "-m", "pip", "freeze"])
if pip_check["returncode"]:
    print("pip check reported dependency conflicts. Read its raw log; the DPO import/hardware check below is required.")

In [ ]:
environment_code = """
import importlib.metadata as metadata
import json
import platform
import resource
import sys
import torch
from trl import DPOConfig, DPOTrainer
from soup_cli.trainer.dpo import DPOTrainerWrapper
from soup_cli.utils.gpu import cuda_supports_bf16

expected = {
    "soup-cli": "0.75.2", "transformers": "5.16.1",
    "trl": "0.29.0", "peft": "0.20.0",
}
actual = {name: metadata.version(name) for name in expected}
gpu = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
report = {
    "python": sys.version, "platform": platform.platform(),
    "versions": actual, "torch": torch.__version__, "cuda_runtime": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu": gpu.name if gpu is not None else None,
    "gpu_total_bytes": gpu.total_memory if gpu is not None else None,
    "compute_capability": [gpu.major, gpu.minor] if gpu is not None else None,
    "native_bf16": cuda_supports_bf16() if gpu is not None else None,
    "host_peak_rss_bytes": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024,
}
print(json.dumps(report, indent=2), flush=True)
assert (3, 10) <= sys.version_info[:2] <= (3, 12), "Soup requires Python 3.10–3.12"
assert actual == expected, "Pinned core package mismatch"
assert gpu is not None and "T4" in gpu.name and (gpu.major, gpu.minor) == (7, 5), "Select a real T4 runtime"
assert not report["native_bf16"], "Recheck T4 fp16 assumptions"
print("DPO imports and T4 checks passed", flush=True)
"""
logged("environment-and-dpo-imports", [sys.executable, "-u", "-c", environment_code], cwd=REPO)
logged("host-memory", ["free", "-h"])
logged("disk-space", ["df", "-h", "/content"])
logged("offline-tests", [sys.executable, "-m", "pytest", "tests", "-q"], cwd=REPO)

## Run the experiment

The runner saves doctor and lint output, the DPO-doctor refusal, the dry run,
streamed/resident parity, training telemetry, adapter verification, a no-op
negative control, and FP16 task/general scores when prerequisites permit them.
It passes measured scores to `soup ship --evidence` at the same FP16 precision.

The current collector exits **1** for incomplete or invalid required evidence,
and **0** for a complete collection. It validates expected nonzero inner exits
against their artifacts: the DPO-doctor refusal, rejected no-op control, adapter
verification reporting **DON'T SHIP**, and Soup's **DON'T SHIP** exit 2. A zero
collector exit can therefore represent a valid measured **DON'T SHIP**.

Read `collection_status`, `failed_commands`, `measured_ship_verdict`, and
`domain_verdict` in `run-summary.json`. Collection completeness and the two
verdicts have separate meanings. The support-domain verdict remains **DON'T
SHIP** until representative support evidence exists.

Preserved attempts 01–03 used a legacy collector that returned zero after
failures; their raw logs, rather than the outer exit, establish what ran.
Attempt 03 stopped at step 33 while logging an infinite scaled AMP gradient
magnitude into JSON. Each retry uses a fresh evidence directory.

In [ ]:
experiment_command = [
    sys.executable, "-u", "scripts/run_experiment.py",
    "--config", "configs/soup.yaml", "--run-dir", str(RUN_DIR),
]
experiment = logged("experiment", experiment_command, cwd=REPO, require_success=False)
summary_path = RUN_DIR / "run-summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print(json.dumps(summary, indent=2))
    collection_status = summary.get("collection_status")
    if collection_status not in ("complete", "incomplete"):
        raise RuntimeError("This summary lacks the current collector contract. Inspect its source and raw logs.")
    expected_exit = 0 if collection_status == "complete" else 1
    assert experiment["returncode"] == expected_exit, "Collector exit disagrees with collection_status"
    print("Collection status:", collection_status)
    print("Measured Soup verdict:", summary.get("measured_ship_verdict"))
    print("Support-domain verdict:", summary.get("domain_verdict"))
else:
    print("Evidence collection is incomplete: no run-summary.json exists. Preserve the experiment raw log.")

for filename in ("streaming-parity.json", "verification.json", "negative-control.json", "ship-verdict.json"):
    report_path = RUN_DIR / filename
    if report_path.exists():
        print(f"\n{filename}\n{report_path.read_text(encoding='utf-8')}")
    else:
        print(f"{filename}: not produced")
print("Experiment collector exit:", experiment["returncode"])

## Export the evidence

Run this cell after success or failure. It includes setup logs and any experiment
artifacts that exist, with the executed source/config/data snapshot and SHA-256
manifest. Model download caches stay outside the archive. Download this notebook
from **File → Download → Download .ipynb** as well to retain its cell outputs.

In [ ]:
if RUN_DIR.exists():
    shutil.copytree(RUN_DIR, EVIDENCE_ROOT / "run", dirs_exist_ok=True)
source_dir = EVIDENCE_ROOT / "source"
source_dir.mkdir(exist_ok=True)
for directory in ("scripts", "configs", "data", "docs", "notebooks", "report", "tests"):
    source = REPO / directory
    if source.exists():
        shutil.copytree(
            source, source_dir / directory, dirs_exist_ok=True,
            ignore=shutil.ignore_patterns("__pycache__", ".ipynb_checkpoints"),
        )
for filename in ("README.md", "requirements-colab.txt", "pytest.ini"):
    source = REPO / filename
    if source.exists():
        shutil.copy2(source, source_dir / filename)

evidence_hashes = {
    path.relative_to(EVIDENCE_ROOT).as_posix(): sha256_file(path)
    for path in sorted(EVIDENCE_ROOT.rglob("*")) if path.is_file()
    and path.name != "evidence-sha256.json"
}
(EVIDENCE_ROOT / "evidence-sha256.json").write_text(
    json.dumps(evidence_hashes, indent=2) + "\n", encoding="utf-8"
)
archive = Path(shutil.make_archive(str(EVIDENCE_ROOT), "zip", EVIDENCE_ROOT))
print("Evidence archive:", archive)
print("Archive SHA-256:", sha256_file(archive))

from google.colab import files
files.download(str(archive))